# Pure fixed V016 postprocessing primitives
No file reads, fits, validation access or submissions on definition. Fit codebook on the actual split training labels only. The fixed policy was selected on A before any E results.


In [ ]:
import numpy as np

def pressure_codebook_v016(training_pressure, training_u_out):
    y=np.asarray(training_pressure,dtype=np.float32)
    phase=np.asarray(training_u_out)
    if y.ndim!=2 or y.shape!=phase.shape or y.shape[1]!=80 or not np.isfinite(y).all() or not np.isin(phase,[0,1]).all():
        raise ValueError("Expected finite complete training breaths and binary phase")
    grid=np.unique(y[phase==0]).astype(np.float64)
    if len(grid)<2:raise ValueError("Insufficient training pressure levels")
    return grid

def weighted_median_midpoint_v016(predictions,weights):
    if set(predictions)!=set(weights) or not weights:raise ValueError("Prediction/weight roster differs")
    w=np.asarray(list(weights.values()),dtype=np.float64)
    if not np.isfinite(w).all() or (w<=0).any() or abs(w.sum()-1)>1e-12:raise ValueError("Invalid fixed weights")
    arrays=[np.asarray(predictions[k]) for k in weights]
    if any(a.ndim!=1 or a.shape!=arrays[0].shape or a.dtype!=np.float64 or not np.isfinite(a).all() for a in arrays):
        raise ValueError("Prediction rows/dtype/finiteness differ")
    values=np.stack(arrays);order=np.argsort(values,axis=0,kind="stable")
    v=np.take_along_axis(values,order,axis=0)
    sw=np.take_along_axis(np.broadcast_to(w[:,None],values.shape),order,axis=0)
    cw=sw.cumsum(0);lo=(cw>=.5).argmax(0);hi=(cw>.5).argmax(0);columns=np.arange(values.shape[1])
    return .5*(v[lo,columns]+v[hi,columns])

def nearest_training_grid_v016(prediction,codebook):
    p=np.asarray(prediction,dtype=np.float64);g=np.asarray(codebook,dtype=np.float64)
    if p.ndim!=1 or g.ndim!=1 or len(g)<2 or not np.isfinite(p).all() or not np.isfinite(g).all() or not (np.diff(g)>0).all():
        raise ValueError("Invalid predictions or training codebook")
    hi=np.searchsorted(g,p).clip(0,len(g)-1);lo=np.maximum(hi-1,0)
    return np.where(np.abs(p-g[lo])<=np.abs(p-g[hi]),g[lo],g[hi])

def selected_postprocessing_v016(predictions,weights,training_codebook):
    return nearest_training_grid_v016(weighted_median_midpoint_v016(predictions,weights),training_codebook)
